# SQL Summative Lab

This notebook completes Part 1 guided SQL queries and Part 2 exploratory analysis for the IMDB movie dataset.


## Part 1: Guided SQL Queries

### Step 1: Connect to Data


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect("data.sqlite")


### Step 2: Limited Edition California Product
Return California customers with a high credit limit above 25,000.


In [ ]:
california_customers = pd.read_sql("""
SELECT 
    customerNumber,
    customerName,
    contactFirstName,
    contactLastName,
    phone,
    city,
    state,
    country,
    creditLimit
FROM customers
WHERE state = 'CA'
  AND CAST(creditLimit AS REAL) > 25000
ORDER BY creditLimit DESC;
""", conn)

california_customers


### Step 3: International Collectable Campaign
Return non-USA customers with “Collect” in the customer name.


In [ ]:
international_collect_customers = pd.read_sql("""
SELECT 
    customerNumber,
    customerName,
    contactFirstName,
    contactLastName,
    city,
    country,
    creditLimit
FROM customers
WHERE country != 'USA'
  AND customerName LIKE '%Collect%'
ORDER BY country, customerName;
""", conn)

international_collect_customers


### Reflection: Step 3

The WHERE clause uses two filters to make sure only the requested customers are selected. The first condition, `country != 'USA'`, removes all customers based in the United States so the results only include international customers. The second condition, `customerName LIKE '%Collect%'`, searches for customer names that contain the word “Collect” anywhere in the name. The `%` symbols mean that other text can appear before or after the word. Using `AND` means both conditions must be true for a customer to appear in the final result.


### Step 4: USA Credit and Inventory Policy - Visual Required
Calculate average credit limit by state for USA-based customers.


In [ ]:
usa_avg_credit_by_state = pd.read_sql("""
SELECT
    state,
    AVG(CAST(creditLimit AS REAL)) AS average_credit_limit,
    COUNT(*) AS number_of_customers
FROM customers
WHERE country = 'USA'
  AND state IS NOT NULL
GROUP BY state
ORDER BY average_credit_limit DESC;
""", conn)

usa_avg_credit_by_state


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(usa_avg_credit_by_state["state"], usa_avg_credit_by_state["average_credit_limit"])
plt.title("Average Customer Credit Limit by State for USA-Based Customers")
plt.xlabel("State")
plt.ylabel("Average Credit Limit")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Step 5: Top Customers - Visual Required
Return top 10 customers by total payment amount.


In [ ]:
top_10_customers = pd.read_sql("""
SELECT
    c.customerName,
    SUM(CAST(p.amount AS REAL)) AS total_payment_amount
FROM customers AS c
JOIN payments AS p
    ON c.customerNumber = p.customerNumber
GROUP BY c.customerNumber, c.customerName
ORDER BY total_payment_amount DESC
LIMIT 10;
""", conn)

top_10_customers


In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(top_10_customers["customerName"], top_10_customers["total_payment_amount"])
plt.title("Top 10 Customers by Total Payment Amount")
plt.xlabel("Total Payment Amount")
plt.ylabel("Customer Name")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Step 6: Top Customer + Product Quantities
Return products each customer purchased in quantities of 10 or more, sorted ascending by total quantity.


In [ ]:
customer_product_quantities = pd.read_sql("""
SELECT
    c.customerName,
    p.productName,
    SUM(od.quantityOrdered) AS total_quantity_purchased
FROM customers AS c
JOIN orders AS o
    ON c.customerNumber = o.customerNumber
JOIN orderdetails AS od
    ON o.orderNumber = od.orderNumber
JOIN products AS p
    ON od.productCode = p.productCode
GROUP BY c.customerName, p.productName
HAVING SUM(od.quantityOrdered) >= 10
ORDER BY total_quantity_purchased ASC;
""", conn)

customer_product_quantities


### Step 7: Product Analysis - Visual Required
Analyze total quantity ordered and product count by product line.


In [ ]:
product_line_analysis = pd.read_sql("""
SELECT
    p.productLine,
    COUNT(DISTINCT p.productCode) AS number_of_products,
    SUM(od.quantityOrdered) AS total_quantity_ordered
FROM products AS p
JOIN orderdetails AS od
    ON p.productCode = od.productCode
GROUP BY p.productLine
ORDER BY total_quantity_ordered DESC;
""", conn)

product_line_analysis


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(
    product_line_analysis["number_of_products"],
    product_line_analysis["total_quantity_ordered"]
)

for i, row in product_line_analysis.iterrows():
    plt.annotate(
        row["productLine"],
        (row["number_of_products"], row["total_quantity_ordered"]),
        fontsize=8
    )

plt.title("Relationship Between Number of Products and Total Quantity Ordered by Product Line")
plt.xlabel("Number of Products in Product Line")
plt.ylabel("Total Quantity Ordered")
plt.tight_layout()
plt.show()


### Reflection: Step 7

I used a scatter plot because the question asks whether product lines with more products also receive more orders. A scatter plot is useful for comparing two numeric measures at the same time: the number of products and the total quantity ordered. Each point represents one product line, so it is easy to see whether the points generally move upward as the number of products increases. In this context, the visual helps show whether larger product lines are also the highest-demand product lines, or whether some smaller product lines still perform strongly.


### Step 8: Remote Offices
Return employees in offices with fewer than 5 employees, including job and supervisor information.


In [ ]:
remote_office_employees = pd.read_sql("""
SELECT
    e.employeeNumber,
    e.firstName,
    e.lastName,
    e.jobTitle,
    e.officeCode,
    o.city AS office_city,
    o.country AS office_country,
    e.reportsTo AS supervisor_employee_number,
    s.firstName || ' ' || s.lastName AS supervisor_name
FROM employees AS e
JOIN offices AS o
    ON e.officeCode = o.officeCode
LEFT JOIN employees AS s
    ON e.reportsTo = s.employeeNumber
WHERE e.officeCode IN (
    SELECT officeCode
    FROM employees
    GROUP BY officeCode
    HAVING COUNT(*) < 5
)
ORDER BY e.officeCode, e.lastName;
""", conn)

remote_office_employees


### Reflection: Step 8

I used the subquery to first identify which office codes have fewer than five employees. The subquery groups the employees by `officeCode` and uses `HAVING COUNT(*) < 5` to keep only the small offices. The main query then uses `WHERE e.officeCode IN (...)` to return only employees who work in those offices. This keeps the query flexible because the office codes are not hard-coded; if employee counts change, the query can still identify the correct offices automatically.


### Step 9: Close the Connection


In [ ]:
conn.close()


## Part 2: Exploratory Analysis with SQL

This section explores the IMDB movie dataset for movies released from 2010 through 2019. The analysis focuses on how genre and runtime relate to movie ratings while excluding null values.


In [ ]:
import zipfile

zip_file_path = 'im.db.zip'
extract_to_path = './'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

conn4 = sqlite3.connect('im.db')


### Check database schema


In [ ]:
schema_df = pd.read_sql("""
SELECT *
FROM sqlite_master;
""", conn4)

schema_df


### Initial data quality check
Check missing values in fields needed for exploration.


In [ ]:
null_check = pd.read_sql("""
SELECT
    SUM(CASE WHEN primary_title IS NULL THEN 1 ELSE 0 END) AS missing_primary_title,
    SUM(CASE WHEN start_year IS NULL THEN 1 ELSE 0 END) AS missing_start_year,
    SUM(CASE WHEN runtime_minutes IS NULL THEN 1 ELSE 0 END) AS missing_runtime,
    SUM(CASE WHEN genres IS NULL THEN 1 ELSE 0 END) AS missing_genres
FROM movie_basics
WHERE start_year BETWEEN 2010 AND 2019;
""", conn4)

null_check


### Movie overview
Summarize the size and time range of the movie dataset.


In [ ]:
movie_overview = pd.read_sql("""
SELECT
    COUNT(*) AS total_movies,
    MIN(start_year) AS earliest_year,
    MAX(start_year) AS latest_year,
    AVG(runtime_minutes) AS average_runtime
FROM movie_basics
WHERE start_year BETWEEN 2010 AND 2019
  AND runtime_minutes IS NOT NULL;
""", conn4)

movie_overview


### Genre rating analysis
Identify top genre categories by average rating, requiring at least 50 movies per genre category so that very small groups do not dominate the result.


In [ ]:
genre_rating_analysis = pd.read_sql("""
SELECT
    mb.genres,
    COUNT(*) AS number_of_movies,
    AVG(mr.averagerating) AS average_rating,
    AVG(mr.numvotes) AS average_votes
FROM movie_basics AS mb
JOIN movie_ratings AS mr
    ON mb.movie_id = mr.movie_id
WHERE mb.start_year BETWEEN 2010 AND 2019
  AND mb.genres IS NOT NULL
  AND mr.averagerating IS NOT NULL
  AND mr.numvotes IS NOT NULL
GROUP BY mb.genres
HAVING COUNT(*) >= 50
ORDER BY average_rating DESC
LIMIT 10;
""", conn4)

genre_rating_analysis


In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(genre_rating_analysis["genres"], genre_rating_analysis["average_rating"])
plt.title("Top Movie Genre Categories by Average Rating, 2010-2019")
plt.xlabel("Average Rating")
plt.ylabel("Genre Category")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Runtime rating analysis
Compare average ratings across runtime groups.


In [ ]:
runtime_rating_analysis = pd.read_sql("""
SELECT
    CASE
        WHEN mb.runtime_minutes < 80 THEN 'Under 80 minutes'
        WHEN mb.runtime_minutes BETWEEN 80 AND 100 THEN '80-100 minutes'
        WHEN mb.runtime_minutes BETWEEN 101 AND 120 THEN '101-120 minutes'
        WHEN mb.runtime_minutes BETWEEN 121 AND 150 THEN '121-150 minutes'
        ELSE 'Over 150 minutes'
    END AS runtime_group,
    COUNT(*) AS number_of_movies,
    AVG(mr.averagerating) AS average_rating
FROM movie_basics AS mb
JOIN movie_ratings AS mr
    ON mb.movie_id = mr.movie_id
WHERE mb.start_year BETWEEN 2010 AND 2019
  AND mb.runtime_minutes IS NOT NULL
  AND mr.averagerating IS NOT NULL
GROUP BY runtime_group
ORDER BY average_rating DESC;
""", conn4)

runtime_rating_analysis


### Part 2: Findings and business question

**Initial exploration findings**
- The dataset contains movies released from 2010 through 2019.
- Some important analysis fields have null values, especially runtime and genre, so those records should be excluded from analyses that depend on those fields.
- Genre categories show differences in average rating, especially when filtering to categories with at least 50 movies.
- Runtime groups can be compared to see whether audiences rate longer or shorter movies differently.

**Business question for deeper analysis**
Which genre categories from 2010-2019 have the strongest combination of high average ratings and enough audience engagement, measured by number of votes, to guide future movie investment or content acquisition decisions?

**Potential data cleaning tasks**
- Exclude rows with null values in fields required for each query, such as `genres`, `runtime_minutes`, `averagerating`, and `numvotes`.
- Split multi-genre values into individual genre tags for more precise genre-level analysis.
- Standardize genre labels and check for inconsistent formatting.
- Exclude movies with `start_year` after 2019 because the assignment notes that those values are not current or accurate.
- Consider vote count thresholds so that a movie or genre with very few votes does not distort average rating conclusions.


### Close the movie database connection


In [ ]:
conn4.close()
